In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd

# Load the dataset
file_path = "/kaggle/input/electricity-demand/final_monthly_dataset.csv"
df = pd.read_csv(file_path)

# Display column names
print(df.columns)
print(df.dtypes)
df.head()

# Data collection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date

# Convert date column to datetime
df['date'] = pd.to_datetime(df['date'])  

# Define the plot layout
fig, ax = plt.subplots(nrows=5, ncols=1, figsize=(15, 25))

# Plot total energy produced
sns.lineplot(x=df['date'], y=df['total_energy_produced'].fillna(np.nan), ax=ax[0], color='dodgerblue')
ax[0].set_title('Feature: Total Energy Produced', fontsize=14)
ax[0].set_ylabel('Total Energy Produced', fontsize=14)

# Plot fuel cost
sns.lineplot(x=df['date'], y=df['total_fuel_cost'].fillna(np.nan), ax=ax[1], color='dodgerblue')
ax[1].set_title('Feature: Total Fuel Cost', fontsize=14)
ax[1].set_ylabel('Total Fuel Cost', fontsize=14)

# Plot temperature
sns.lineplot(x=df['date'], y=df['apparent_temperature_mean (°C)'].fillna(np.nan), ax=ax[2], color='dodgerblue')
ax[2].set_title('Feature: Apparent Temperature', fontsize=14)
ax[2].set_ylabel('Temperature (°C)', fontsize=14)

# Plot fuel quantity
sns.lineplot(x=df['date'], y=df['total_fuel_quantity'].fillna(np.nan), ax=ax[3], color='dodgerblue')
ax[3].set_title('Feature: Total Fuel Quantity', fontsize=14)
ax[3].set_ylabel('Fuel Quantity', fontsize=14)

# Plot Demand (Target)
sns.lineplot(x=df['date'], y=df['Demand'].fillna(np.nan), ax=ax[4], color='red')
ax[4].set_title('Target: Demand', fontsize=14)
ax[4].set_ylabel('Demand', fontsize=14)

# Set x-axis limits
for i in range(5):
    ax[i].set_xlim([date(2019, 1, 1), date(2024, 12, 30)])

plt.show()


# Handling missing values

In [ ]:
# Check for missing values in each column
missing_values = df.isna().sum()

# Display columns with missing values
missing_values[missing_values > 0]

*Series([], dtype: int64) means that there are no missing values in any of the columns*

# Feature Scaling MinMax Scaling (0 to 1)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
numerical_cols = ['total_energy_produced', 'day_peak', 'evening_peak', 'Demand', 'Supply', 'total_fuel_cost', 'apparent_temperature_mean (°C)', 'total_fuel_quantity']
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df.drop(columns=['area', 'fuel_type', 'season'], inplace=True)
df.head()

# PCA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Load dataset
file_path = "/kaggle/input/electricity-demand/final_monthly_dataset.csv"
df = pd.read_csv(file_path)

# Select only numerical columns for PCA (Exclude 'date', 'area', 'fuel_type', 'season')
numeric_features = df.select_dtypes(include=[np.number])

# Handle missing values by filling with mean
numeric_features = numeric_features.fillna(numeric_features.mean())

# Standardize the data (important for PCA)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(numeric_features)

# Apply PCA
pca = PCA(n_components=len(numeric_features.columns))  # Use all features
pca.fit(scaled_data)

# Get feature importance (PCA loadings)
loadings = pd.DataFrame(pca.components_.T, columns=[f"PC{i+1}" for i in range(len(numeric_features.columns))], index=numeric_features.columns)

# Plot feature importance (heatmap)
plt.figure(figsize=(10, 6))
sns.heatmap(loadings, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Feature Importance in PCA (Loadings)")
plt.xlabel("Principal Components")
plt.ylabel("Features")
plt.show()

# Print explained variance to see how much information each PC captures
explained_variance = pca.explained_variance_ratio_
print("Explained Variance by Principal Components:")
for i, var in enumerate(explained_variance):
    print(f"PC{i+1}: {var:.4f} ({var*100:.2f}% of variance)")

# Feature Importance Based on Loadings
# Absolute value of loadings for all principal components (important features will have larger values)
importance_df = loadings.abs()

# Sum the absolute loadings to get the overall importance of each feature
feature_importance = importance_df.sum(axis=1)

# Sort features by their importance
sorted_importance = feature_importance.sort_values(ascending=False)

# Normalize feature importance
normalized_importance = feature_importance / feature_importance.sum()

# Print the sorted normalized feature importance
print("\nNormalized Feature Importance (Sorted):")
print(normalized_importance.sort_values(ascending=False))




# Augmented Dickey-Fuller (ADF) Test to check stationary

In [ ]:
from statsmodels.tsa.stattools import adfuller

# Perform ADF test
demand_series = df['Demand']
result_demand = adfuller(demand_series)

# Extract values for readability
adf_statistic = result_demand[0]
p_value = result_demand[1]
critical_values = result_demand[4]

# Print results
print(f"ADF Statistic: {adf_statistic}")
print(f"p-value: {p_value}")
print(f"Critical Values: {critical_values}")

# Interpretation
if p_value > 0.05:
    print("The data is non-stationary (fail to reject null hypothesis).")
else:
    print("The data is stationary (reject null hypothesis).")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

f, ax = plt.subplots(nrows=3, ncols=2, figsize=(15, 9))

def visualize_adfuller_results(series, title, ax):
    result = adfuller(series)
    significance_level = 0.05
    adf_stat = result[0]
    p_val = result[1]
    crit_val_1 = result[4]['1%']
    crit_val_5 = result[4]['5%']
    crit_val_10 = result[4]['10%']

    if (p_val < significance_level) & (adf_stat < crit_val_1):
        linecolor = 'forestgreen'  # Stationary at 1% level
    elif (p_val < significance_level) & (adf_stat < crit_val_5):
        linecolor = 'gold'  # Stationary at 5% level
    elif (p_val < significance_level) & (adf_stat < crit_val_10):
        linecolor = 'orange'  # Stationary at 10% level
    else:
        linecolor = 'indianred'  # Not stationary

    sns.lineplot(x=df['date'], y=series, ax=ax, color=linecolor)
    ax.set_title(f'ADF Statistic {adf_stat:0.3f}, p-value: {p_val:0.3f}\nCritical Values 1%: {crit_val_1:0.3f}, 5%: {crit_val_5:0.3f}, 10%: {crit_val_10:0.3f}', fontsize=14)
    ax.set_ylabel(ylabel=title, fontsize=14)

# Visualizing ADF test results for your dataset columns
visualize_adfuller_results(df['total_energy_produced'], 'Total Energy Produced', ax[0, 0])
visualize_adfuller_results(df['day_peak'], 'Day Peak', ax[0, 1])
visualize_adfuller_results(df['evening_peak'], 'Evening Peak', ax[1, 0])
visualize_adfuller_results(df['Demand'], 'Demand', ax[1, 1])
visualize_adfuller_results(df['Supply'], 'Supply', ax[2, 0])

# Remove the empty subplot
f.delaxes(ax[2, 1])

# Adjust layout and show the plot
plt.tight_layout()
plt.show()

# Exploratory Data Analysis 
**plotting the seasonal components of each feature and comparing the minima and maxima.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose

# Convert 'date' column to datetime if it's not already
df['date'] = pd.to_datetime(df['date'])

# List of columns to decompose
decompose_cols = ['total_energy_produced', 'day_peak', 'evening_peak', 'Demand', 'Supply']

# Create subplots for 5 features
f, ax = plt.subplots(nrows=5, ncols=1, figsize=(15, 12))

# Set a common title for the plots
f.suptitle('Seasonal Components of Features', fontsize=16)

# Perform seasonal decomposition for each column and plot the seasonal components
for i, col in enumerate(decompose_cols):
    # Perform seasonal decomposition
    result = seasonal_decompose(df[col], model='additive', period=52, extrapolate_trend='freq')
    
    # Plot the seasonal component
    sns.lineplot(x=df['date'], y=result.seasonal, ax=ax[i], color='dodgerblue', label=col)
    ax[i].set_ylabel(ylabel=col, fontsize=14)
    ax[i].set_title(f'Seasonal Component of {col}', fontsize=14)

# Convert date limits to pandas datetime format
start_date = pd.to_datetime("2019-01-01")
end_date = pd.to_datetime("2024-12-30")

# Set the x-axis limits for all subplots
for i in range(5):
    ax[i].set_xlim([start_date, end_date])

# Adjust layout to prevent overlap and show the plot
plt.tight_layout()
plt.show()


# correlation matrix

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# List of numeric columns for correlation analysis (removed duplicate 'Supply' and 'district')
correlation_cols = ['total_energy_produced', 'day_peak', 'evening_peak', 'Demand', 'Supply', 'fuel', 'total_fuel_quantity']

# Compute the correlation matrix for the selected numeric columns
corr_matrix = df[correlation_cols].corr()

# Plot the correlation matrix using a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation Matrix of Selected Features', fontsize=16)
plt.tight_layout()
plt.show()


# ACF and PACF

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt

# Select the column to analyze
col = 'Demand'  # Replace with the desired column

# Set appropriate lag length for yearly seasonality (22 months for your specific case)
seasonal_lag = 22  # For monthly data, assuming a custom lag

# Create a figure with two subplots
plt.figure(figsize=(16, 8))

# Plot ACF (Autocorrelation Function) on the left side
plt.subplot(1, 2, 1)  # (rows, columns, index)
plot_acf(df[col], lags=seasonal_lag*2, ax=plt.gca(), color='dodgerblue')
plt.title(f'ACF of {col}', fontsize=16)

# Plot PACF (Partial Autocorrelation Function) on the right side
plt.subplot(1, 2, 2)  # (rows, columns, index)
plot_pacf(df[col], lags=seasonal_lag*2, ax=plt.gca(), color='dodgerblue')
plt.title(f'PACF of {col}', fontsize=16)

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()


In [ ]:
print(df.columns)

# Bi-LSTM

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score
from math import sqrt
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Bidirectional
from tensorflow.keras.optimizers import Adam


# Ensure all required columns are present
numeric_cols = ['total_energy_produced', 'day_peak', 'evening_peak', 'Demand', 
                'Supply', 'apparent_temperature_mean (°C)', 'total_fuel_quantity', 'district']

df = df[numeric_cols].dropna()  # Drop missing values

# Select features (X) and target (y)
X = df.drop(columns=['Demand']).values  # Convert to NumPy array
y = df['Demand'].values.reshape(-1, 1)  # Reshape target for scaling

# Normalize features and target
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

# Reshape the data for LSTM input [samples, time steps, features]
X_scaled = X_scaled.reshape(X_scaled.shape[0], 1, X_scaled.shape[1])

# Split data into training (80%) and testing (20%)
train_size = int(len(df) * 0.8)
X_train, X_test = X_scaled[:train_size], X_scaled[train_size:]
y_train, y_test = y_scaled[:train_size], y_scaled[train_size:]

# Build the Bi-LSTM model
model = Sequential()

# Bi-LSTM layer
model.add(Bidirectional(LSTM(units=64, return_sequences=False), input_shape=(X_train.shape[1], X_train.shape[2])))

# Output layer
model.add(Dense(1))

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

# Train the model
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test), verbose=1)

# Predict using the trained model
predictions = model.predict(X_test)

# Inverse transform predictions and actual values
predictions = scaler_y.inverse_transform(predictions)
y_test_inv = scaler_y.inverse_transform(y_test)

# Save the trained model
model.save('demand_forecast_Bi-LSTM_model.h5')

# Print model summary
model.summary()

# Evaluate the model
mape = mean_absolute_percentage_error(y_test_inv, predictions)
rmse = sqrt(mean_squared_error(y_test_inv, predictions))
r2 = r2_score(y_test_inv, predictions)

# Print evaluation metrics
print(f"MAPE: {mape:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")

# Plot actual vs predicted values
plt.figure(figsize=(10, 6))
plt.plot(range(train_size, len(df)), y_test_inv, label='Actual Demand', color='blue')
plt.plot(range(train_size, len(df)), predictions, label='Predicted Demand', color='orange')
plt.title('Bi-LSTM Model Prediction vs Actual Demand')
plt.xlabel('Time')
plt.ylabel('Demand')
plt.legend()
plt.show()


# RNN

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score
from math import sqrt
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense
from tensorflow.keras.optimizers import Adam

# Ensure all required columns are present
numeric_cols = ['total_energy_produced', 'day_peak', 'evening_peak', 'Demand', 
                'Supply', 'apparent_temperature_mean (°C)', 'total_fuel_quantity', 'district']

df = df[numeric_cols].dropna()  # Drop missing values

# Select features (X) and target (y)
X = df.drop(columns=['Demand']).values  # Convert to NumPy array
y = df['Demand'].values.reshape(-1, 1)  # Reshape target for scaling

# Normalize features and target
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

# Reshape the data for RNN input [samples, time steps, features]
X_scaled = X_scaled.reshape(X_scaled.shape[0], 1, X_scaled.shape[1])

# Split data into training (80%) and testing (20%)
train_size = int(len(df) * 0.8)
X_train, X_test = X_scaled[:train_size], X_scaled[train_size:]
y_train, y_test = y_scaled[:train_size], y_scaled[train_size:]

# Build the RNN model
model = Sequential()

# Add RNN layer
model.add(SimpleRNN(units=50, activation='relu', input_shape=(X_train.shape[1], X_train.shape[2])))

# Add output layer
model.add(Dense(1))

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

# Train the model
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test), verbose=1)

# Predict using the trained model
predictions = model.predict(X_test)

# Inverse transform predictions and actual values
predictions = scaler_y.inverse_transform(predictions)
y_test_inv = scaler_y.inverse_transform(y_test)

# Save the trained model
model.save('demand_forecast_RNN_model.h5')

# Print model summary
model.summary()

# Evaluate the model
mape = mean_absolute_percentage_error(y_test_inv, predictions)
rmse = sqrt(mean_squared_error(y_test_inv, predictions))
r2 = r2_score(y_test_inv, predictions)

# Print evaluation metrics
print(f"MAPE: {mape:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")

# Plot actual vs predicted values
plt.figure(figsize=(10, 6))
plt.plot(range(train_size, len(df)), y_test_inv, label='Actual Demand', color='blue')
plt.plot(range(train_size, len(df)), predictions, label='Predicted Demand', color='orange')
plt.title('RNN Model Prediction vs Actual Demand ')
plt.xlabel('Time')
plt.ylabel('Demand')
plt.legend()
plt.show()


# CNN-LSTM

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score
from math import sqrt
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Ensure all required columns are present
numeric_cols = ['total_energy_produced', 'day_peak', 'evening_peak', 'Demand', 
                'Supply', 'apparent_temperature_mean (°C)', 'total_fuel_quantity', 'district']

df = df[numeric_cols].dropna()  # Drop missing values

# Select features (X) and target (y)
X = df.drop(columns=['Demand']).values  # Convert to NumPy array
y = df['Demand'].values.reshape(-1, 1)  # Reshape target for scaling

# Normalize features and target
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

# Reshape the data for CNN-LSTM input [samples, time steps, features]
X_scaled = X_scaled.reshape(X_scaled.shape[0], 1, X_scaled.shape[1])

# Split data into training (80%) and testing (20%)
train_size = int(len(df) * 0.8)
X_train, X_test = X_scaled[:train_size], X_scaled[train_size:]
y_train, y_test = y_scaled[:train_size], y_scaled[train_size:]

# Build the CNN-LSTM model
model = Sequential()

# Build CNN-LSTM model
model = Sequential([
    # 1D Convolution Layer with kernel_size=1
    Conv1D(filters=64, kernel_size=1, activation='relu', input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    
    # LSTM Layer
    LSTM(50, activation='relu', return_sequences=False),
    Dropout(0.2),
    
    # Dense Layer for output
    Dense(1)
])


# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

# Train the model
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test), verbose=1)

# Predict using the trained model
predictions = model.predict(X_test)

# Inverse transform predictions and actual values
predictions = scaler_y.inverse_transform(predictions)
y_test_inv = scaler_y.inverse_transform(y_test)

# Save the trained model
model.save('demand_forecast_CNN_LSTM_model.h5')

# Print model summary
model.summary()

# Evaluate the model
mape = mean_absolute_percentage_error(y_test_inv, predictions)
rmse = sqrt(mean_squared_error(y_test_inv, predictions))
r2 = r2_score(y_test_inv, predictions)

# Print evaluation metrics
print(f"MAPE: {mape:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")

# Plot actual vs predicted values
plt.figure(figsize=(10, 6))
plt.plot(range(train_size, len(df)), y_test_inv, label='Actual Demand', color='blue')
plt.plot(range(train_size, len(df)), predictions, label='Predicted Demand', color='orange')
plt.title('CNN-LSTM Model Prediction vs Actual Demand ')
plt.xlabel('Time')
plt.ylabel('Demand')
plt.legend()
plt.show()


# RNN-LSTM

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score
from math import sqrt
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Ensure all required columns are present
numeric_cols = ['total_energy_produced', 'day_peak', 'evening_peak', 'Demand', 
                'Supply', 'apparent_temperature_mean (°C)', 'total_fuel_quantity', 'district']

df = df[numeric_cols].dropna()  # Drop missing values

# Select features (X) and target (y)
X = df.drop(columns=['Demand']).values  # Convert to NumPy array
y = df['Demand'].values.reshape(-1, 1)  # Reshape target for scaling

# Normalize features and target
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

# Reshape the data for RNN-LSTM input [samples, time steps, features]
X_scaled = X_scaled.reshape(X_scaled.shape[0], 1, X_scaled.shape[1])

# Split data into training (80%) and testing (20%)
train_size = int(len(df) * 0.8)
X_train, X_test = X_scaled[:train_size], X_scaled[train_size:]
y_train, y_test = y_scaled[:train_size], y_scaled[train_size:]

# Build RNN-LSTM model
model = Sequential([
    # RNN Layer
    SimpleRNN(64, activation='relu', return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    
    # LSTM Layer
    LSTM(50, activation='relu', return_sequences=False),
    Dropout(0.2),
    
    # Dense Layer for output
    Dense(1)
])

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

# Train the model
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test), verbose=1)

# Predict using the trained model
predictions = model.predict(X_test)

# Inverse transform predictions and actual values
predictions = scaler_y.inverse_transform(predictions)
y_test_inv = scaler_y.inverse_transform(y_test)

# Save the trained model
model.save('demand_forecast_RNN_LSTM_model.h5')

# Print model summary
model.summary()

# Evaluate the model
mape = mean_absolute_percentage_error(y_test_inv, predictions)
rmse = sqrt(mean_squared_error(y_test_inv, predictions))
r2 = r2_score(y_test_inv, predictions)

# Print evaluation metrics
print(f"MAPE: {mape:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")

# Plot actual vs predicted values
plt.figure(figsize=(10, 6))
plt.plot(range(train_size, len(df)), y_test_inv, label='Actual Demand', color='blue')
plt.plot(range(train_size, len(df)), predictions, label='Predicted Demand', color='orange')
plt.title('RNN-LSTM Model Prediction vs Actual Demand ')
plt.xlabel('Time')
plt.ylabel('Demand')
plt.legend()
plt.show()


# XGBoost

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score
from math import sqrt
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import train_test_split

# Ensure all required columns are present
numeric_cols = ['total_energy_produced', 'day_peak', 'evening_peak', 'Demand', 
                'Supply', 'apparent_temperature_mean (°C)', 'total_fuel_quantity', 'district']

df = df[numeric_cols].dropna()  # Drop missing values

# Select features (X) and target (y)
X = df.drop(columns=['Demand']).values  # Convert to NumPy array
y = df['Demand'].values  # Target variable

# Normalize features and target
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()  # Flatten target after scaling

# Split data into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

# Initialize XGBoost Regressor
model = xgb.XGBRegressor(
    objective='reg:squarederror',  # Regression task
    n_estimators=100,  # Number of boosting rounds
    max_depth=6,  # Maximum depth of tree
    learning_rate=0.1,  # Learning rate
    subsample=0.8,  # Subsample ratio
    colsample_bytree=0.8,  # Column sampling ratio
    n_jobs=-1,  # Number of threads used in computation
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

# Predict using the trained model
predictions = model.predict(X_test)

# Inverse transform predictions and actual values
predictions_inv = scaler_y.inverse_transform(predictions.reshape(-1, 1))
y_test_inv = scaler_y.inverse_transform(y_test.reshape(-1, 1))

# Save the trained model
model.save_model('demand_forecast_XGBoost_model.json')

# Evaluate the model
mape = mean_absolute_percentage_error(y_test_inv, predictions_inv)
rmse = sqrt(mean_squared_error(y_test_inv, predictions_inv))
r2 = r2_score(y_test_inv, predictions_inv)

# Print evaluation metrics
print(f"MAPE: {mape:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")

# Plot actual vs predicted values
plt.figure(figsize=(10, 6))
plt.plot(range(len(y_test_inv)), y_test_inv, label='Actual Demand', color='blue')
plt.plot(range(len(predictions_inv)), predictions_inv, label='Predicted Demand', color='orange')
plt.title('XGBoost Model Prediction vs Actual Demand ')
plt.xlabel('Time')
plt.ylabel('Demand')
plt.legend()
plt.show()
